# Imports 

In [1]:
import pandas as pd

from fragility.config import DEMO_DIR
from fragility.universe import load_universe, load_holdings
from fragility.portfolio import load_portfolio
from fragility.lookthrough import (
    company_exposure, sector_exposure, country_exposure, overlap_matrix,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## Load the reference data 

In [2]:
universe = load_universe()
holdings = load_holdings()

print(f"{len(universe)} funds in the universe, "
      f"{holdings['etf_ticker'].nunique()} with holdings data")

20 funds in the universe, 20 with holdings data


## Load the overlapping demo portfolio 

In [3]:
result = load_portfolio(DEMO_DIR / "overlapping.csv", universe)
portfolio = result.portfolio

print("Errors: ", result.errors or "none")
print("Notices:", result.notices or "none")
portfolio

c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

Errors:  none
Notices: none


,ticker,name,type,value_gbp,weight,holding_id
0,VWRL.L,Vanguard FTSE All-World,etf,6000.0,0.24,VWRL.L
1,VUSA.L,Vanguard S&P 500 (Dist),etf,5000.0,0.20,VUSA.L
2,EQQQ.L,Invesco EQQQ Nasdaq-100,etf,5000.0,0.20,EQQQ.L
3,CSPX.L,iShares Core S&P 500,etf,4000.0,0.16,CSPX.L
4,IITU.L,iShares S&P 500 IT Sector,etf,2000.0,0.08,IITU.L
5,AAPL,Apple Inc.,share,1500.0,0.06,APPLE
6,NVDA,NVIDIA Corporation,share,1500.0,0.06,NVIDIA


## the top 10 companies (the actual checklist item)

In [4]:
exposure = company_exposure(portfolio, holdings, universe)
companies = exposure[exposure["kind"] == "company"]   # leave out the "Other" bucket
top10 = companies.head(10)

show = top10[["company", "exposure", "exposure_gbp", "via"]].copy()
show["exposure"] = (show["exposure"] * 100).round(1).astype(str) + "%"
show["exposure_gbp"] = show["exposure_gbp"].map("£{:,.0f}".format)
show.index = range(1, len(show) + 1)
show

,company,exposure,exposure_gbp,via
1,NVIDIA Corp,13.4%,"£3,338","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
2,Apple Inc,12.9%,"£3,237","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
3,Microsoft Corp,5.4%,"£1,358","CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
4,Alphabet Inc,3.7%,£913,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
5,Amazon.com Inc,2.7%,£671,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
6,Micron Technology Inc,2.1%,£533,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
7,Advanced Micro Devices Inc,1.8%,£460,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
8,Broadcom Inc,1.8%,£446,"CSPX.L, IITU.L, VUSA.L, VWRL.L"
9,Meta Platforms Inc,1.8%,£439,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
10,Tesla Inc,1.5%,£369,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"


## Check 

In [5]:
residual = exposure.loc[exposure["kind"] == "residual", "exposure"].sum()
blocks = exposure[exposure["kind"] == "block"]
multi = companies[companies["n_routes"] >= 2]

print(f"Total exposure:          {exposure['exposure'].sum():.4f}   (must be 1.0000)")
print(f"Top 10 companies:        {top10['exposure'].sum():.1%} of your money "
      f"(£{top10['exposure_gbp'].sum():,.0f} of £{portfolio['value_gbp'].sum():,.0f})")
print(f"Funds' smaller holdings: {residual:.1%}")
print(f"Reached via 2+ routes:   {len(multi)} companies")
print("Funds not looked through:", blocks["company"].tolist() or "none")

Total exposure:          1.0000   (must be 1.0000)
Top 10 companies:        47.1% of your money (£11,764 of £25,000)
Funds' smaller holdings: 29.2%
Reached via 2+ routes:   59 companies
Funds not looked through: none


## Sectors, countries and fund overlap 

In [6]:
display(sector_exposure(portfolio, holdings, universe).head(5))
display(country_exposure(portfolio, holdings, universe).head(5))

funds = portfolio.loc[portfolio["type"] == "etf", "ticker"].tolist()
overlap_matrix(funds, holdings).round(2)

,sector,exposure,exposure_gbp,via
0,Information Technology,0.463442,11586.055,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct"
1,Unclassified (smaller holdings),0.292109,7302.721,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
2,Communication Services,0.059268,1481.695,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
3,Consumer Discretionary,0.044448,1111.196,"CSPX.L, EQQQ.L, VUSA.L, VWRL.L"
4,Unknown,0.040768,1019.195,EQQQ.L


,country,exposure,exposure_gbp,via
0,United States,0.641643,16041.075,"CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L"
1,Unknown,0.221375,5534.376,"VWRL.L, direct"
2,Mixed (smaller holdings),0.136295,3407.376,VWRL.L
3,Switzerland,0.000687,17.178,VWRL.L


,VWRL.L,VUSA.L,EQQQ.L,CSPX.L,IITU.L
VWRL.L,1.00,0.37,0.26,0.36,0.19
VUSA.L,0.37,1.00,0.43,0.59,0.33
EQQQ.L,0.26,0.43,1.00,0.41,0.41
CSPX.L,0.36,0.59,0.41,1.00,0.32
IITU.L,0.19,0.33,0.41,0.32,1.00


In [7]:
import pandas as pd
from fragility.config import ETF_UNIVERSE_CSV, SCENARIOS_CSV

u = pd.read_csv(ETF_UNIVERSE_CSV, dtype=str).fillna("")
u["first_date"] = pd.to_datetime(u["first_date"])
earliest = pd.Timestamp(pd.read_csv(SCENARIOS_CSV)["start_date"].min())
first = u.set_index("ticker")["first_date"]

print(f"Oldest scenario starts {earliest.date()}\n")
for r in u[u["first_date"] > earliest].itertuples():
    if not r.proxy:
        verdict = "NO PROXY - needs one"
    elif r.proxy not in first.index:
        verdict = f"proxy {r.proxy} is not in the universe"
    elif first[r.proxy] > earliest:
        verdict = f"proxy {r.proxy} is too new itself"
    else:
        verdict = f"proxy {r.proxy} ok"
    print(f"{r.ticker:8} starts {r.first_date.date()}  ->  {verdict}")

Oldest scenario starts 2016-06-23

VUAG.L   starts 2019-05-14  ->  proxy VUSA.L ok
VAGP.L   starts 2019-06-18  ->  proxy IGLT.L ok


In [8]:
from fragility.prices import load_prices
from fragility.stress import (load_scenarios, tickers_needed, replay_all,
                              scenario_table, worst_scenario, run_hypotheticals)

scenarios = load_scenarios()
prices = load_prices(tickers_needed(portfolio, universe))
results = replay_all(portfolio, prices, scenarios, universe)

table = scenario_table(results)
for col in ("return", "max_drawdown"):
    table[col] = table[col].map("{:+.1%}".format)
table["loss_gbp"] = table["loss_gbp"].map("£{:,.0f}".format)
display(table[["scenario", "dates", "return", "loss_gbp", "max_drawdown",
               "worst_contributors", "estimated"]])

worst = worst_scenario(results)
print(f"Worst: {worst.name}, {worst.worst_loss:.1%}")
run_hypotheticals(portfolio, holdings, universe)

c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

,scenario,dates,return,loss_gbp,max_drawdown,worst_contributors,estimated
0,Covid crash,2020-02-19 to 2020-03-23,-23.8%,"£-5,956",-23.8%,"VWRL.L, VUSA.L, CSPX.L",
1,2022 rate shock,2021-12-09 to 2022-06-16,-20.1%,"£-5,013",-20.1%,"EQQQ.L, VWRL.L, VUSA.L",
2,Q4 2018 sell-off,2018-09-03 to 2018-12-24,-19.7%,"£-4,925",-19.8%,"EQQQ.L, VWRL.L, NVDA",
3,UK mini-budget,2022-09-22 to 2022-10-14,-4.8%,"£-1,202",-7.6%,"EQQQ.L, VWRL.L, NVDA",
4,Brexit vote,2016-06-23 to 2016-06-27,+5.0%,"£1,254",+0.0%,,


Worst: Covid crash, -23.8%


,scenario,exposed,loss,loss_gbp
0,US tech -20%,0.288069,-0.057614,-1440.347
1,UK equities -15%,0.000000,-0.000000,-0.000


In [9]:
from fragility.config import DEMO_DIR
from fragility.pipeline import analyse_csv
from fragility.alerts import split_alerts

r = analyse_csv(DEMO_DIR / "overlapping.csv")
shown, more = split_alerts(r.alerts)
for a in shown:
    print(f"[{'RED' if a.severity == 3 else 'AMBER' if a.severity == 2 else 'OK'}] "
          f"{a.title}: {a.message}")
print(f"\n+{len(more)} more findings")

c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
c:\Users\pakhi\OneDrive\Documents\Projects\fragility-monitor\fragility\Lib\site-packages\yfinance\utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#d

[RED] Illusion of choice: You hold 7 investments, but they move like 1.3 groups.
[RED] Hidden single-company bet: NVIDIA Corp is 13% of your money (£3,338), through 6 different routes: CSPX.L, EQQQ.L, IITU.L, VUSA.L, VWRL.L, direct. 1 other company is similar.
[AMBER] Top-heavy: 47% of your money sits in just 10 companies (£11,764).

+1 more findings


In [10]:
from fragility.prices import load_prices

p = load_prices(["HMWO.L", "SWDA.L"])          # SWDA tracks the same index
moves = p.pct_change(fill_method=None)
big = moves["HMWO.L"].abs().sort_values(ascending=False).head(3)
print("Biggest HMWO daily moves:\n", big.round(3), "\n")

for day in big.index:
    window = p.loc[day - pd.Timedelta(days=6): day + pd.Timedelta(days=6)]
    print(f"--- around {day.date()} ---")
    print(window.round(2), "\n")

Biggest HMWO daily moves:
 Date
2025-10-24    0.343
2025-10-27    0.243
2020-03-12    0.076
Name: HMWO.L, dtype: float64 

--- around 2025-10-24 ---
Ticker      HMWO.L  SWDA.L
Date                      
2025-10-20   32.53   95.12
2025-10-21   32.62   95.36
2025-10-22   32.48   94.98
2025-10-23   32.70   95.64
2025-10-24   43.92  128.38
2025-10-27   33.23   97.20
2025-10-28   33.50   97.96
2025-10-29   33.66   98.46
2025-10-30   33.70   98.43 

--- around 2025-10-27 ---
Ticker      HMWO.L  SWDA.L
Date                      
2025-10-21   32.62   95.36
2025-10-22   32.48   94.98
2025-10-23   32.70   95.64
2025-10-24   43.92  128.38
2025-10-27   33.23   97.20
2025-10-28   33.50   97.96
2025-10-29   33.66   98.46
2025-10-30   33.70   98.43
2025-10-31   33.56   98.18 

--- around 2020-03-12 ---
Ticker      HMWO.L  SWDA.L
Date                      
2020-03-06   16.37   44.08
2020-03-09   15.39   41.48
2020-03-10   15.38   41.38
2020-03-11   15.40   41.46
2020-03-12   14.23   38.24
2020-03-13  